In [11]:
import pandas as pd
import requests

from rdkit import Chem
from rdkit.Chem import AllChem  


# Get name for solvent smiles

no need to run this section again

In [6]:
df = pd.read_csv('/home/nanta/Solv_GNN_SSD/data/solvent_smiles.csv')

In [7]:
def get_name_from_smiles(smiles):
    # Replace special characters for URL encoding
    smiles_encoded = smiles.replace("#", "%23").replace("+", "%2B").replace("/", "%2F").replace("\\", "%5C")
    
    # Construct the URL for PubChem API
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smiles_encoded}/property/IUPACName/JSON"
    
    # Send a request to the PubChem API
    response = requests.get(url)
    
    # Check if the response is successful
    if response.status_code == 200:
        data = response.json()
        try:
            return data['PropertyTable']['Properties'][0]['IUPACName']
        except (KeyError, IndexError):
            return "No name found for this SMILES."
    else:
        return "Error: " + str(response.status_code)

In [8]:
smile = df['can_smiles_solvent'].iloc[0]
name = get_name_from_smiles(smile)
print(f"SMILES: {smile}\nIUPAC Name: {name}")

SMILES: CCCC
IUPAC Name: butane


In [9]:
df['solvent_name'] = df['can_smiles_solvent'].apply(get_name_from_smiles)
df.to_csv('/home/nanta/Solv_GNN_SSD/data/solvent_smiles_with_names.csv', index=False)


# Predict free energy of solvation

In [12]:
smile = "C(CCC(=O)O)CCN"
# get canonical smiles
mol = Chem.MolFromSmiles(smile)
can_smile = Chem.MolToSmiles(mol, canonical=True)
print(can_smile)

NCCCCCC(=O)O


In [14]:
# create input file for GNN
df = pd.read_csv('/home/nanta/Solv_GNN_SSD/data/solvent_smiles_with_names.csv')
df['can_smiles_solute'] = can_smile
df['DGsolv'] = [None] * len(df)
df.to_csv('/home/nanta/Solv_GNN_SSD/data/Solvent_query/To_query.csv', index=False)

In [ ]:
# run on bash
# 1) cd to Solv_GNN_SSD folder
# 2) activate the conda environment
#   "conda activate tf24gpu"
# 3) run the following command
# "python main.py -filename [FILENAME, e.g., /home/nanta/Solv_GNN_SSD/data/Solvent_query/To_query.csv] -predict_df -modelname SSD_models/student35

In [ ]:
# df2 = pd.read_csv('/home/nanta/Solv_GNN_SSD/data/Solvent_query/Solvent_search_aminocaproic_acid_results.csv')
# df2['solvent_name'] = df['solvent_name']
# df2.to_csv('/home/nanta/Solv_GNN_SSD/data/Solvent_query/Solvent_search_aminocaproic_acid_results.csv', index=False)